# AI Image Generator with Multilingual Support

**Generate stunning images from text descriptions in any language!**

## Key Features
- **Multilingual Support**: Automatic translation of any language to English
- **Free GPU**: Uses Google Colab's free GPU acceleration
- **Advanced Controls**: Adjustable inference steps, guidance scale, and image dimensions
- **Multiple Prompts**: Support for positive and negative prompts

## Setup Instructions
1. This notebook **requires GPU acceleration**
2. Go to `Runtime` → `Change runtime type` → Select `T4 GPU` or better
3. Run cells from top to bottom

## How It Works
Your input → Automatic Translation to English → Image Generation using Stable Diffusion XL → Display Result

## Configuration Constants

All important settings and model configurations are defined below for easy customization.

In [ ]:
# ============================================================================
# Configuration Constants
# ============================================================================

# Model configurations
TRANSLATION_MODEL = "facebook/m2m100_418M"
IMAGE_GENERATION_MODEL = "Lykon/dreamshaper-xl-lightning"
TARGET_LANGUAGE = "en"

# Generation parameter ranges
INFERENCE_STEPS_MIN = 1
INFERENCE_STEPS_MAX = 50
INFERENCE_STEPS_DEFAULT = 25
GUIDANCE_SCALE_MIN = 1.0
GUIDANCE_SCALE_MAX = 15.0
GUIDANCE_SCALE_DEFAULT = 7.5
IMAGE_SIZES = [512, 768, 1024]
DEFAULT_IMAGE_SIZE = 768

# Translation settings
MAX_TRANSLATION_TOKENS = 128

# Logging
VERBOSE = True  # Set to False to reduce console output

## STEP 1: Install Required Libraries

Install all dependencies needed for image generation and translation. This may take 1-2 minutes on first run.

In [ ]:
# Install required packages (silently)
# - diffusers: Hugging Face diffusion models pipeline
# - transformers: Pre-trained transformer models for translation
# - torch, torchvision: PyTorch deep learning framework
# - accelerate: Distributed training utilities
# - safetensors: Safe model serialization format
# - gradio: Optional - for web UI in production

print("Installing required packages...")
!pip install --quiet diffusers transformers torch torchvision accelerate safetensors
print("All packages installed successfully!")

## STEP 2: Authenticate with Hugging Face (Optional)

Login to Hugging Face Hub to access gated models. This step is recommended but optional.

In [ ]:
# Step 1: Import required authentication libraries
from huggingface_hub import login
from google.colab import userdata

# Step 2: Retrieve HF token from Colab Secrets
# To add your HF_TOKEN:
# 1. Go to https://huggingface.co/settings/tokens
# 2. Create or copy your token
# 3. In Colab, click Secrets in the left sidebar
# 4. Add a new secret: name="HF_TOKEN", value=<your_token>

try:
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        login(hf_token, add_to_git_credential=True)
        print("Successfully logged in to Hugging Face Hub")
    else:
        print("HF_TOKEN not found - proceeding without authentication")
        print("Note: Some models may have access restrictions")
except Exception as e:
    print(f"Authentication skipped: {str(e)}")
    print("Note: You can continue, but some models may require login")

## STEP 3: Import Libraries and Check GPU

Import all required Python libraries and verify GPU availability.

In [ ]:
# Import core libraries for image generation and translation
import torch
from diffusers import StableDiffusionXLPipeline
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display

# Import translation libraries
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

# ============================================================================
# GPU Detection and Information
# ============================================================================

print("=" * 60)
print("GPU INFORMATION")
print("=" * 60)

# Check GPU availability
gpu_available = torch.cuda.is_available()
print(f"GPU Available: {gpu_available}")

if gpu_available:
    device_name = torch.cuda.get_device_name(0)
    device_memory = torch.cuda.get_device_properties(0).total_memory / 1e9  # Convert to GB
    print(f"GPU Name: {device_name}")
    print(f"GPU Memory: {device_memory:.1f} GB")
    device = "cuda"
else:
    print("No GPU detected - Using CPU (slower)")
    device = "cpu"

print("=" * 60)

In [ ]:
# ============================================================================
# STEP 4: Define Translation Function
# ============================================================================

def translate_to_english(text: str) -> str:
    """
    Automatically translate any language input to English using M2M100 model.
    
    Supports 100+ languages with automatic language detection.
    Falls back to returning original text if translation fails.
    
    Args:
        text (str): Input text in any language
        
    Returns:
        str: Translated text in English, or original text if empty/error
    """
    
    # Skip translation for empty strings
    if not text or not text.strip():
        return text
    
    if VERBOSE:
        print(f"[Translation] Processing: '{text[:60]}...'")
    
    try:
        # Step 1: Load tokenizer for the translation model
        tokenizer = M2M100Tokenizer.from_pretrained(TRANSLATION_MODEL)
        
        # Step 2: Load the M2M100 model with GPU optimization if available
        if torch.cuda.is_available():
            model = M2M100ForConditionalGeneration.from_pretrained(
                TRANSLATION_MODEL,
                torch_dtype=torch.float16
            ).to("cuda")
        else:
            model = M2M100ForConditionalGeneration.from_pretrained(TRANSLATION_MODEL)
        
        # Step 3: Tokenize input text
        inputs = tokenizer(text, return_tensors="pt")
        
        # Step 4: Move inputs to GPU if available
        if torch.cuda.is_available():
            inputs = {k: v.to("cuda") for k, v in inputs.items()}
        
        # Step 5: Set target language to English
        target_lang_id = tokenizer.get_lang_id(TARGET_LANGUAGE)
        
        # Step 6: Generate translation
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                forced_bos_token_id=target_lang_id,
                max_new_tokens=MAX_TRANSLATION_TOKENS
            )
        
        # Step 7: Decode and return translated text
        translated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        if VERBOSE:
            print(f"[Translation] Result: '{translated_text}'")
        
        return translated_text
        
    except Exception as e:
        # Log error and return original text as fallback
        print(f"[Warning] Translation failed: {str(e)}")
        print(f"[Fallback] Using original text: '{text}'")
        return text

## STEP 5: Load Image Generation Model

Load the Stable Diffusion XL model for high-quality image generation. This may take 2-3 minutes on first run.

In [ ]:
# ============================================================================
# Load Stable Diffusion XL Model
# ============================================================================

print("=" * 60)
print("LOADING IMAGE GENERATION MODEL")
print("=" * 60)
print(f"Model: {IMAGE_GENERATION_MODEL}")
print(f"Device: {device}")
print("This may take 2-3 minutes on first run...")
print("=" * 60)

try:
    # Load pre-trained Stable Diffusion XL model
    pipe = StableDiffusionXLPipeline.from_pretrained(
        IMAGE_GENERATION_MODEL,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        safety_checker=None,  # Disable safety checker for faster inference
    )
    
    # Move pipeline to device
    pipe = pipe.to(device)
    
    # Optimization: Enable attention slicing to reduce memory usage
    if device == "cuda":
        pipe.enable_attention_slicing()
    
    print("Model loaded successfully!")
    print(f"Using device: {device}")
    print("=" * 60)
    
except Exception as e:
    print(f"Error loading model: {str(e)}")
    print("Please check your GPU availability and retry")
    raise

## STEP 6: Define Image Generation Helper Function

Create a reusable function to generate and display images with any language input.

In [ ]:
def generate_and_display_image(
    prompt: str,
    negative_prompt: str = "",
    steps: int = INFERENCE_STEPS_DEFAULT,
    guidance_scale: float = GUIDANCE_SCALE_DEFAULT,
    width: int = DEFAULT_IMAGE_SIZE,
    height: int = DEFAULT_IMAGE_SIZE,
    seed: int = -1,
    save_path: str = None
):
    """
    Generate an image from text prompt and display it.
    
    Automatically translates non-English prompts to English before generation.
    
    Args:
        prompt (str): Positive prompt in any language
        negative_prompt (str): Negative prompt (features to avoid)
        steps (int): Number of inference steps (higher = better quality but slower)
        guidance_scale (float): CFG scale for guidance strength
        width (int): Image width in pixels
        height (int): Image height in pixels
        seed (int): Random seed (-1 for random)
        save_path (str): Path to save image (optional)
        
    Returns:
        PIL.Image: Generated image
    """
    
    print("=" * 60)
    print("IMAGE GENERATION PROCESS")
    print("=" * 60)
    
    # Step 1: Translate prompts to English
    print("[1/4] Translating prompts to English...")
    translated_prompt = translate_to_english(prompt)
    translated_negative = translate_to_english(negative_prompt) if negative_prompt.strip() else ""
    
    # Step 2: Set up random seed
    print("[2/4] Setting up generation parameters...")
    if seed >= 0:
        generator = torch.Generator(device=device).manual_seed(int(seed))
    else:
        generator = None
    
    # Step 3: Generate image
    print(f"[3/4] Generating image ({steps} steps, guidance_scale={guidance_scale})...")
    output = pipe(
        prompt=translated_prompt,
        negative_prompt=translated_negative,
        num_inference_steps=int(steps),
        guidance_scale=float(guidance_scale),
        height=int(height),
        width=int(width),
        generator=generator
    )
    image = output.images[0]
    
    # Step 4: Display and save image
    print("[4/4] Displaying result...")
    
    # Display with matplotlib
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(image)
    ax.axis('off')
    
    # Add title with original prompt
    title = f"Prompt: {prompt}\n"
    if negative_prompt:
        title += f"Negative: {negative_prompt}"
    ax.set_title(title, fontsize=11, wrap=True, pad=20)
    
    plt.tight_layout()
    plt.show()
    
    # Save image if path provided
    if save_path:
        image.save(save_path)
        print(f"Image saved to: {save_path}")
    
    print("=" * 60)
    print("Generation completed!")
    print("=" * 60)
    
    return image

## STEP 7: Three examples


In [ ]:
# Example 1: English Prompt
prompt_en = "A majestic white wolf with glowing crystal blue eyes, standing on a snowy mountain peak at night, hyperrealistic, 8k resolution, mystical atmosphere"

# Generate image
image_1 = generate_and_display_image(
    prompt=prompt_en,
    steps=25,
    guidance_scale=7.5,
    save_path="generated_image_english.png"
)

In [ ]:
# Example 2: Vietnamese Prompt
prompt_vi = "Một con mèo phi hành gia đang ôm mặt trăng, phong cách hoạt hình 3D đáng yêu"

# Generate image (will be automatically translated to English)
image_2 = generate_and_display_image(
    prompt=prompt_vi,
    steps=25,
    guidance_scale=7.5,
    save_path="generated_image_vietnamese.png"
)

In [ ]:
# Example 3: Japanese Prompt
prompt_ja = "サイバーパンクの夜の街、ネオンライト、高いビル、アニメスタイル"

# Generate image (will be automatically translated to English)
image_3 = generate_and_display_image(
    prompt=prompt_ja,
    steps=25,
    guidance_scale=7.5,
    save_path="generated_image_japanese.png"
)